# Block 4 lecture — documents that are not tables yet

**The lecture's queries, as Eduardo ran them: for review after class. Nothing here is typed in class.** Most
sections were *predict, then watch*: the slide showed the code and a question, you wrote your answer and one reason
in your notes, and the next slide showed the cell and what it printed. Sections 1, 3, 6, 8 and 9 were *watch*: shown and explained, with no
prediction. The sections are numbered as the slides number them.

It reads the World Bank pages in `data/raw/api/worldbank/` and the three-entry `data/raw/api/toy_rates.json`. It never
opens the lab's file. At home, cover each output, predict it, then run the cell.

In [ ]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

## Section 1 — One request, live (the slide showed this call as captured on 25 September 2026)

A public server is shared by everyone. Sixty laptops asking the same question at once is not polite, and the answer
is already saved in `data/raw/api/worldbank/`. At home, with a network, you may run it once; without one, the cell
says so, and section 2 reads the same answer from the cache.

In [ ]:
import urllib.request

url = ("https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"
       "?format=json&date=2010:2024&per_page=1000&page=1")
request = urllib.request.Request(url, headers={"User-Agent": "ecbs5294-lecture"})   # say who is asking
try:
    with urllib.request.urlopen(request, timeout=10) as response:                    # never wait forever
        print("status:", response.status)
        live = json.load(response)
    print(live[0])
except OSError as problem:            # no Wi-Fi, a timeout, a refusal: the cached page below is the same answer
    print("No live answer:", problem, "-> use the cached page")

## Section 2 — The cached page — predict, then watch

Open page 1 from the cache. Predict: what kind of thing is it, how long is it, and what is its first element?

In [ ]:
page1 = json.load(open("data/raw/api/worldbank/SP.POP.TOTL_page1.json"))
print(type(page1), len(page1))
print(page1[0])
print(len(page1[1]), "records on this page")

## Section 3 — What DuckDB sees — watch

`read_json_auto` on the same file, and the JSON type of each row DuckDB made.

In [ ]:
con.sql("""
    SELECT json_type(json) AS element
    FROM read_json_auto('data/raw/api/worldbank/SP.POP.TOTL_page1.json')
""").df()

## Section 4 — The list inside the document becomes rows — predict, then watch

`unnest` turns a list into one row per element. `->` goes one level into a JSON object; `->>` takes the value out
as text. Predict: is Hungary on this page? Write one reason.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE population_page1 AS
    SELECT record->>'countryiso3code'       AS iso3,
           record->'country'->>'value'      AS country,
           CAST(record->>'date' AS INTEGER) AS year,
           CAST(record->>'value' AS BIGINT) AS population
    FROM (SELECT unnest(CAST(json AS JSON[])) AS record          -- one row per element of the list
          FROM read_json_auto('data/raw/api/worldbank/SP.POP.TOTL_page1.json')
          WHERE json_type(json) = 'ARRAY')                       -- element 0 is the metadata: skip it
""")
con.sql("""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT country) AS n_countries,
           COUNT(*) FILTER (WHERE country = 'Hungary') AS hungary_rows
    FROM population_page1
""").df()

## Section 5 — Every page — predict, then watch

The metadata says how many pages there are. Loop over them, collect every record, and count against `total`.
Predict: what would it print **without the `+ 1`**? Write one reason.

In [ ]:
records = []
for page in range(1, page1[0]["pages"] + 1):
    doc = json.load(open(f"data/raw/api/worldbank/SP.POP.TOTL_page{page}.json"))
    records += doc[1]
print(len(records), "records; the metadata says", page1[0]["total"])

### Section 5b — The same loop without the `+ 1` (the slide's question)

In [ ]:
records_short = []
for page in range(1, page1[0]["pages"]):          # no + 1: range stops before its end
    doc = json.load(open(f"data/raw/api/worldbank/SP.POP.TOTL_page{page}.json"))
    records_short += doc[1]
print(len(records_short), "records without the + 1; the metadata says", page1[0]["total"])

## Section 6 — The same in SQL, all four files at once — watch

A `*` in the path reads every page. The count against `total` again, and Hungary in 2024. Section 11 uses this
table, `population`.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE population AS
    SELECT record->>'countryiso3code'       AS iso3,
           record->'country'->>'value'      AS country,
           CAST(record->>'date' AS INTEGER) AS year,
           CAST(record->>'value' AS BIGINT) AS population
    FROM (SELECT unnest(CAST(json AS JSON[])) AS record
          FROM read_json_auto('data/raw/api/worldbank/SP.POP.TOTL_page*.json')
          WHERE json_type(json) = 'ARRAY')
""")
print(con.sql("SELECT COUNT(*), COUNT(DISTINCT country) FROM population").fetchone())
con.sql("SELECT * FROM population WHERE country = 'Hungary' AND year = 2024").df()

## Section 7 — A dictionary of dictionaries — predict, then watch

`toy_rates.json` holds three real days copied from an ECB rates response. Predict: how many rows does DuckDB make
of it, and what type is `rates`? Write one reason.

In [ ]:
print(open("data/raw/api/toy_rates.json").read())

### Section 7b — What DuckDB made of it

In [ ]:
con.sql("DESCRIBE SELECT * FROM read_json_auto('data/raw/api/toy_rates.json')").df()

## Section 8 — Struct access — watch

To ask for one day, the query has to write the day as a name.

In [ ]:
con.sql("""
    SELECT base, rates."2017-01-02".BRL AS brl
    FROM read_json_auto('data/raw/api/toy_rates.json')
""").df()

## Section 9 — The loop — watch

The dates are the keys, so they belong in a column. Six lines of Python, and one to show the table.

In [ ]:
toy = json.load(open("data/raw/api/toy_rates.json"))
rows = []
for day, values in toy["rates"].items():
    rows.append((day, values["BRL"]))
con.execute("CREATE OR REPLACE TABLE toy_rates (date DATE, value DOUBLE)")
con.executemany("INSERT INTO toy_rates VALUES (?, ?)", rows)
con.sql("SELECT * FROM toy_rates").df()

## Section 10 — What is each number? — after the slide's question

The slide asked: if the base is EUR, what is 3.4265, reais for one euro or euros for one real? And 100 reais on that
day are how many euros?

In [ ]:
toy = json.load(open("data/raw/api/toy_rates.json"))
print("base:", toy["base"], "| amount:", toy["amount"])
for day, values in toy["rates"].items():
    print(day, values["BRL"], "->", f"{100 / values['BRL']:.2f}", "euros for 100 reais")

## Section 11 — The rate of a whole is a weighted mean — predict, then watch

Real rows from section 6's `population` table: Brazil and Hungary, 2010 and 2024. Brazil is about twenty times
Hungary's size; Brazil grew, Hungary shrank. Predict: did the two countries **together** grow by the plain average of
their two growths? If not, is the real figure nearer Brazil's growth or Hungary's? Write one reason. And would a check
"between the lowest and the highest growth" tell the two figures apart?

In [ ]:
together, plain, lowest, highest = con.sql("""
    WITH g AS (
        SELECT country,
               MAX(CASE WHEN year = 2010 THEN population END) AS pop_2010,
               MAX(CASE WHEN year = 2024 THEN population END) AS pop_2024
        FROM population
        WHERE country IN ('Brazil', 'Hungary')
        GROUP BY country
    )
    SELECT SUM(pop_2024) / SUM(pop_2010) AS together,        -- each country at its own growth
           AVG(pop_2024 / pop_2010)      AS plain_average,   -- the two growths, averaged
           MIN(pop_2024 / pop_2010)      AS lowest,
           MAX(pop_2024 / pop_2010)      AS highest
    FROM g
""").fetchone()
print(f"together {together:.4f} | plain average {plain:.4f} | lowest {lowest:.4f} | highest {highest:.4f}")

## Section 12a — A share of a total, in one `SELECT` — predict, then watch

Each country's share of the pair's 2010 population: its population over the two added up. Predict: two shares, or an
error? Write one reason. The `try` prints DuckDB's refusal and keeps *Run All* going.

In [ ]:
try:
    con.sql("""
    SELECT country,
           population / SUM(population) AS share
    FROM population
    WHERE year = 2010 AND country IN ('Brazil', 'Hungary')
""").df()
except duckdb.Error as e:
    print("DuckDB refused:", str(e).splitlines()[0])

### Section 12b — The total as a query inside the query

The query in brackets runs once and gives one number, the pair's total; every row is divided by it. Its `WHERE` is
the outer query's `WHERE`, so the shares add up to 1. These shares are section 11's weights.

In [ ]:
con.sql("""
    SELECT country, population,
           population / (SELECT SUM(population) FROM population
                         WHERE year = 2010 AND country IN ('Brazil', 'Hungary')) AS share
    FROM population
    WHERE year = 2010 AND country IN ('Brazil', 'Hungary')
""").df()

---

## What to remember from Block 4

1. **An API's answer is a document, not a table.** Ask with a timeout and a name, check the status code, then check
   the answer: `200` means the server answered, not that it answered what you meant (page 9 of 4 came back `200`,
   with an empty list).
2. **Every JSON shape hides a table.** A list of records → rows. A dictionary keyed by id → the key becomes a
   column. A dictionary of dictionaries → (outer key, inner key, value). A list inside a record → a child table, with
   a key that points back to its parent.
3. **One table per grain.** Two grains in one document make two tables and a key that links them: population by
   country and year; the country's name and region once per country.
4. **Pagination.** Read `pages` and `total` from the metadata, fetch every page, and count what you got against
   `total` (3,975 = 3,975). Page 1 alone was 1,000 records, and it looked exactly like success.
5. **DuckDB reads JSON** — `read_json_auto`, `unnest` for a list, `->` and `->>` for fields, a dot for a struct.
   When the keys are data (dates), a six-line Python loop that turns keys into rows is clearer.
6. **Offline-first.** Save the answer under `data/raw/api/`, write the URL, the date and the SHA-256 in `DATA.md`,
   and read the file from then on. Asking again is a decision, and you say so.
7. **A rate has a direction, and the document says which.** `amount: 1.0, base: EUR` makes `"BRL": 3.4265` *reais for
   one euro*: 100 reais are 100 / 3.4265 = 29.18 euros. The key's name says which currency, not per what.
8. **The rate of a whole is a weighted mean of its parts' rates, weighted by their size.** Brazil and Hungary
   together grew by a factor of 1.0877 from 2010 to 2024; the plain average of their growths is 1.0253, which would
   lose about 12.7 million people. Both lie between the lowest growth (0.9562) and the highest (1.0945), so that range
   bound is plausibility, not proof. A converted total works the same way: euros over reais is the daily rates
   averaged with the reais as weights.
9. **The proof is a line you compute yourself** — one amount, one day, by hand from the document, agreeing with the
   table to the cent. And when rows can lose their partner in a join, count the missing conversions (the NULLs), not
   the rows that joined.
10. **A share of a total is a part over its whole, and the whole is a query inside the query.**
    `population / SUM(population)` in one `SELECT` is refused: `SUM` makes one row, `country` has two (Block 2's
    `GROUP BY` rule). `population / (SELECT SUM(population) FROM … WHERE …)` divides every row by one number. Give the
    inner query the outer query's `WHERE`, and the shares add up to 1: Brazil 0.9509, Hungary 0.0491 in 2010 — the
    weights of section 11 (0.9509 × 1.0945 + 0.0491 × 0.9562 = 1.0877).

## Common mistakes

- **Treating `200` as success.** A misspelled World Bank indicator and a page that does not exist both came back
  `200`. Read the answer, not only the status.
- **Stopping at page 1, or at page 3.** `range(1, pages)` has no `+ 1`, so it stops before the last page: 3,000
  records of 3,975, and nothing errors. The count against `total` is what catches it.
- **Filtering on `->>` without parentheses.** In DuckDB 1.5.5 `->>` binds more loosely than `AND`:
  `j->>'a' = 'x' AND j->>'b' = 'y'` stops with a conversion error. Wrap each one, `(j->>'a') = 'x'`, or better, pull
  the fields out into named columns first and filter the columns.
- **Single quotes in struct access.** `rates.'2017-01-02'` is a string, not a field. A name that starts with a digit
  needs double quotes: `rates."2017-01-02"`.
- **Indentation is syntax.** The `append` line belongs under the `for`. Take its indent away and Python stops
  before running anything: `IndentationError: expected an indented block after 'for' statement`. The quiet version
  is worse: if another line stays inside the loop and the `append` moves out, it runs once, after the loop has
  finished, and the table has one row, the last day, with no error.
- **Reading the direction from the key's name.** `BRL` says which currency the number is counted in; `base` and
  `amount` say *per what*. Multiplying 100 reais by 3.4265 gives 342.65 — a number of the right shape, about twelve
  times too big.
- **Averaging rates as if every part were the same size.** The plain average of Brazil's and Hungary's growth is
  1.0253; the two together grew 1.0877. A figure for the whole is weighted by the size of each part.
- **Offering the range bound as the proof.** A converted total at the wrong rates can sit inside the range; only the
  one-line hand calculation tells right from wrong.
- **A share whose total has a different filter.** If the query in brackets keeps rows the outer query does not (another
  year, every country), the shares still run, and no longer add up to 1. Add them up: that is the check.